# Exercise 8: Hyperpameter tuning/optimization: Random and Grid Search

Pima Indian Diabetes dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

The datasets consists of several medical predictor variables and one target variable, _Outcome_. Predictor variables includes the number of pregnancies the patient has had, their BMI, insulin level, age, and so on.

[Pima Indian Diabes dataset in Kaggle](https://www.kaggle.com/uciml/pima-indians-diabetes-database)

Dataset can found from Moodle -> Dataset.

__This exercises HAVE TO be done in CSC Puhti cloud environment.__

__NOTE: Try to get as good as possible accuracy. Accurary is one of the evalution criteria__.

TOP results will be found from [here](https://student.labranet.jamk.fi/~rmika/DLNN/TOP-scores.htm)



## Grid Search

Lots of code examples can be found from [this tutorial](https://machinelearningmastery.com/grid-search-hyperparameters-deep-learning-models-python-keras/) for different hyperparameters.

### Use scikit-learn to grid search the batch size, epochs, dropout_rate and learning_rate

In [1]:
# Downloading reqiuired libraries
%pip install tensorflow
%pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import libraries

import os

# Limit TensorFlow to the allocated SLURM CPUs on Roihu
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", os.environ.get("SLURM_CPUS_PER_TASK", "1"))
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")

# Optional: Suppress the TF INFO/WARNING messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from tensorflow import keras
from scikeras.wrappers import KerasClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
SEED = 42
keras.utils.set_random_seed(SEED)

# Set the number of jobs for parallel processing
SEARCH_JOBS = 1

I0000 00:00:1793259994.285798 3837126 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793259994.301977 3837126 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793259998.685532 3837126 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793259998.695572 3837126 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
# Dataset splitting

# Load the dataset
data = pd.read_csv("pima-indians-diabetes.csv", header=None)
data.info()

# Extract features and labels
features = data.iloc[:, :8].astype(float).copy()
labels = data.iloc[:, 8]

# Handle missing values and later impute them using SimpleImputer in the pipeline
# Zero denotes missing measurements in five columns (Glucose, Blood Pressure, Skin Thickness, Insulin, BMI)
# Pregnancy count and Outcomes column retain their zero values
# Age and Diabetes Pedigree Function columns do not have zero values
features.iloc[:, [1, 2, 3, 4, 5]] = features.iloc[:, [1, 2, 3, 4, 5]].replace(0, np.nan)

# X and Y are only the training data; the final test set stays outside both searches
X, X_test, Y, Y_test = train_test_split(
    features, labels, test_size=0.20,
    stratify=labels, random_state=SEED
)

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Define the grid search parameters as follows:
# learning_rate:[0.001, 0.01, 0.1]
# batch_size: [10, 40 , 80]
# number of epochs: [10, 50, 100]
# dropout_rate:  [0.0, 0.3, 0.5]

# Define the hyperparameter grid for the search as asked in the exercise instructions
learning_rate = [0.001, 0.01, 0.1]
batch_size = [10, 40, 80]
epochs = [10, 50, 100]
dropout_rate = [0.0, 0.3, 0.5]

# NOTE: If you don't have enough memory in your computer, drops some parameters

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       768 non-null    int64  
 1   1       768 non-null    int64  
 2   2       768 non-null    int64  
 3   3       768 non-null    int64  
 4   4       768 non-null    int64  
 5   5       768 non-null    float64
 6   6       768 non-null    float64
 7   7       768 non-null    int64  
 8   8       768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
#Create model
    #Input layer = 8
    #Only one hidden layer = 12 Dense nodes, activation function = relu
    #Dropout layer
    #Output layer: 1 node, activation = sigmoid

def create_model(learning_rate=0.001, dropout_rate=0.0):
    keras.backend.clear_session()                                   # Clear previous models from memory to avoid clutter and potential conflicts
    inputs = keras.Input(shape=(8,))                                #Input layer = 8
    hidden = keras.layers.Dense(12, activation="relu")(inputs)      #Only one hidden layer = 12 Dense nodes, activation function = relu
    hidden = keras.layers.Dropout(dropout_rate)(hidden)             #Dropout layer
    outputs = keras.layers.Dense(1, activation="sigmoid")(hidden)   #Output layer: 1 node, activation = sigmoid 
    
    network = keras.Model(inputs=inputs, outputs=outputs)
    
    network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    
    return network

#Define KerasClassifier
classifier = KerasClassifier(
    model=create_model, model__learning_rate=0.001, model__dropout_rate=0.0,
    random_state=SEED, verbose=0
)

# Each CV fold learns its own medians, means, and standard deviations.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("nn", classifier)
])

In [5]:
#Set parameter dictionary
param_grid = dict(
    nn__model__learning_rate=learning_rate,
    nn__batch_size=batch_size,
    nn__epochs=epochs,
    nn__model__dropout_rate=dropout_rate
)
#Define grid = GridSearcCV(....)
grid = GridSearchCV(
    estimator=model, param_grid=param_grid, n_jobs=SEARCH_JOBS, cv=cv,
    scoring="accuracy", verbose=1, refit=True, error_score="raise"
)
print(f"Grid search: {len(learning_rate) * len(batch_size) * len(epochs) * len(dropout_rate)} candidates, 3 folds each.")


Grid search: 81 candidates, 3 folds each.


In [6]:
#Results
grid_result = grid.fit(X, Y)

Fitting 3 folds for each of 81 candidates, totalling 243 fits


In [7]:
#Summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))


Best: 0.786641 using {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.5, 'nn__model__learning_rate': 0.01}
0.767097 (0.014042) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.001}
0.771983 (0.021970) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.01}
0.765487 (0.013545) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.1}
0.760553 (0.014935) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.001}
0.776893 (0.018159) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.01}
0.750821 (0.018154) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.1}
0.747537 (0.008883) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__mo

## Random Search

In [8]:
# Import libraries
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

#Create model
#Same as in Grid search example 
# Reuse create_model through a fresh clone of the same preprocessing/model pipeline.

#Define KerasClassifier
random_model = clone(model)


In [9]:
# define the Random search parameters
# randomly from same range as the Grid search

#Set parameter dictionary
param_distributions = param_grid.copy()
N_ITER = 20

#Define random_search = RandomizedSearchCV(...)
random_search = RandomizedSearchCV(
    estimator=random_model, param_distributions=param_distributions,
    n_iter=N_ITER, scoring="accuracy", n_jobs=SEARCH_JOBS, cv=cv,
    random_state=SEED, verbose=1, refit=True, error_score="raise"
)
print(f"Random search: {N_ITER} sampled candidates, 3 folds each.")


Random search: 20 sampled candidates, 3 folds each.


In [10]:
#Results
random_search.fit(X, Y)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=0))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'nn__batch_size': [10, 40, ...], 'nn__epochs': [10, 50, ...], 'nn__model__dropout_rate': [0.0, 0.3, ...], 'nn__model__learning_rate': [0.001, 0.01, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies

In [11]:
#Summarize results
print("Best: %f using %s" % (random_search.best_score_, random_search.best_params_))
means = random_search.cv_results_['mean_test_score']
stds = random_search.cv_results_['std_test_score']
params = random_search.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))


Best: 0.783365 using {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 40}
0.719863 (0.002953) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 10, 'nn__batch_size': 40}
0.767097 (0.014042) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 10}
0.742667 (0.018008) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 100, 'nn__batch_size': 10}
0.775251 (0.022103) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 10, 'nn__batch_size': 40}
0.776869 (0.018001) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 100, 'nn__batch_size': 10}
0.783365 (0.015408) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 40}
0.754065 (0.028028) with: {'nn__model__learning_rate': 0.01, 'nn__mod

In [12]:
# Final check on the held-out test set (not used in the searches)
for name, search in [("Grid search", grid_result), ("Random search", random_search)]:
    pred = search.best_estimator_.predict(X_test)
    print(f"{name}: CV accuracy {search.best_score_:.3f}, test accuracy {accuracy_score(Y_test, pred):.3f}")

Grid search: CV accuracy 0.787, test accuracy 0.695
Random search: CV accuracy 0.783, test accuracy 0.727


# Conclusion

__NOTE: Conclusion is one evaluation criterion for the exercise, so pay attention to writing this__

Write conclusion:




In this exercised we were asked to tune the hyperparameters of a small neural network on the Pima Indians diabetes data, using grid search (all 81 combinations of learning rate, batch size, epochs and dropout rate) and random search (20 random combinations from the same ranges). 

I used 3-fold cross-validation, and the data preparation (filling missing values and scaling) was done inside the pipeline.

| | Best CV accuracy | Best settings | Test accuracy |
|---|---|---|---|
| Grid search (243 fits) | 0.787 | lr 0.01, batch 10, 10 epochs, dropout 0.5 | 0.695 |
| Random search (60 fits) | 0.783 | lr 0.01, batch 40, 10 epochs, dropout 0.0 | 0.727 |

Both methods found almost the same accuracy, but random search needed only 60 fits instead of
243, so it was much cheaper. The learning rate had the clearest effect (0.01 worked best on
average). Epochs, batch size and dropout made little difference, and many combinations gave
almost the same result.

The test accuracy is lower than the cross-validation accuracy. The best cross-validation score
is a bit optimistic, because it is the best of many tries, and the test set is small, so its
accuracy can change by several points.

### What I learned

Grid search tries everything and takes a long time, while random search is faster and often good
enough. The data has to be prepared inside the cross-validation to avoid leakage, and a separate
test set is useful for checking the result.


### AI Usage

I used AI as an assistant in this exercise. It helped me check my code against the exercise,
remove unused code, and understand the results, for example why the test accuracy was lower
than the cross-validation accuracy. It also helped me draft the conclusion. I ran the notebook
myself and checked that the conclusion matches my results.